In [23]:
# Import required packages 
import pandas as pd
import numpy as np 
import pypdf
import re
import requests
import json



In [24]:
# Global Variables 
# Set headers to accept JSON response
HEADERS = {'Accept': 'application/json'}
# File location 
pdf_file_path = 'Desktop/13023_2024_Article_3213.pdf' # Replace with the path to your PDF file

# Hgnc rest api address 
HGNC_REST = "https://rest.genenames.org/fetch/symbol"
# Ensemble rest api addresses 
ENSEMBL_REST_38 = "https://rest.ensembl.org"
ENSEMBL_REST_19 = "https://grch37.rest.ensembl.org"

# Ontology look up rest api address
OLS_BASE = "https://www.ebi.ac.uk/ols4/api"


In [25]:
# Function for reading in text from a pdf file

def extract_text_from_pdf(pdf_path):
    # Open the PDF file in binary read mode
    with open(pdf_path, 'rb') as file:
        # Create a PdfReader object
        reader = pypdf.PdfReader(file)
        
        # Initialize a string to store all extracted text
        full_text = ""
        
        # Get the total number of pages
        num_pages = len(reader.pages)
        print(f"Number of pages: {num_pages}")
        
        # Iterate through each page
        for page_num in range(num_pages):
            # Get the specific page object
            page = reader.pages[page_num]
            
            # Extract text from the page and append it to the full_text string
            # The extract_text() method returns a string with the text content
            full_text += page.extract_text() or "" # Use or "" to handle cases where no text is extracted

    #return full text of pdf file        
    return full_text

extracted_content = extract_text_from_pdf(pdf_file_path)

####### TESTING ##################
# Print or save the extracted text
#print(extracted_content) 
# To save to a text file:
# with open('output.txt', 'w', encoding='utf-8') as output_file:
#     output_file.write(extracted_content)


Number of pages: 9


In [26]:
def likely_gene_symbol(s: str) -> bool:
    s = str(s).strip()
    return bool(re.fullmatch(r"[A-Z][A-Z0-9]{2,11}", s))

In [27]:
def check_hgnc_symbol(symbol):
    # Construct the API URL for a gene symbol search
    url = f"{HGNC_REST}/{symbol}"

    

    try:
        response = requests.get(url, headers=HEADERS)
        response.raise_for_status() # Raise an exception for bad status codes

        data = response.json()

        #print(data)

        docs = data.get("response", {}).get("docs", [])

        if not docs:
            print(f"Gene '{symbol}' not found.")
            return None


        gene_info = docs[0]

        gene = {
            "hgnc_id": gene_info.get("hgnc_id"),
            "symbol": gene_info.get("symbol"),
            "aliases": gene_info.get("alias_symbol", []),
            "prev_symbols": gene_info.get("prev_symbol", []),
            "entrez_id": gene_info.get("entrez_id"),
            "ensembl_id": gene_info.get("ensembl_gene_id"),
            "omim_ids": gene_info.get("omim_id", [])
        }


        # Check if any documents (matches) were returned
        #if data['response']['numFound'] > 0:
            # The first doc is the primary match for the symbol
        #    gene_info = data['response']['docs'][0]

            
            
        print(f"Gene '{symbol}' found. HGNC ID: {gene_info['hgnc_id']}, Name: {gene_info['symbol']}")
        #else:
        #    print(f"Gene '{symbol}' not found.")
        return gene

    except requests.exceptions.RequestException as e:
        print(f"An error occurred during the API request: {e}")
        return None

####### TESTING ##################
# Example usage:
practice_hgnc = check_hgnc_symbol("BRCA1")
print(practice_hgnc)
check_hgnc_symbol("InvalidGeneSymbol")

#print("....found genes....")



Gene 'BRCA1' found. HGNC ID: HGNC:1100, Name: BRCA1
{'hgnc_id': 'HGNC:1100', 'symbol': 'BRCA1', 'aliases': ['RNF53', 'BRCC1', 'PPP1R53', 'FANCS'], 'prev_symbols': [], 'entrez_id': '672', 'ensembl_id': 'ENSG00000012048', 'omim_ids': ['113705']}
Gene 'InvalidGeneSymbol' not found.


In [28]:
def check_ensembl_location(symbol,url_hg19):
    # Construct the API URL for a gene symbol search
    url = f"https://rest.genenames.org/fetch/symbol/{symbol}"

    if url_hg19 == True: 
        url  = f"https://grch37.rest.ensembl.org/fetch/symbol/{symbol}"
     
    


    
  

    try:
        response = requests.get(url, headers=HEADERS)
        response.raise_for_status() # Raise an exception for bad status codes

        data = response.json()

        #print(data)

        docs = data.get("response", {}).get("docs", [])

        if not docs:
            print(f"Gene '{symbol}' not found.")
            return None


        gene_info = docs[0]

        gene = {
            "hgnc_id": gene_info.get("hgnc_id"),
            "symbol": gene_info.get("symbol"),
            "aliases": gene_info.get("alias_symbol", []),
            "prev_symbols": gene_info.get("prev_symbol", []),
            "entrez_id": gene_info.get("entrez_id"),
            "ensembl_id": gene_info.get("ensembl_gene_id"),
            "omim_ids": gene_info.get("omim_id", [])
        }


        # Check if any documents (matches) were returned
        #if data['response']['numFound'] > 0:
            # The first doc is the primary match for the symbol
        #    gene_info = data['response']['docs'][0]

            
            
        print(f"Gene '{symbol}' found. HGNC ID: {gene_info['hgnc_id']}, Name: {gene_info['symbol']}")
        #else:
        #    print(f"Gene '{symbol}' not found.")
        return gene

    except requests.exceptions.RequestException as e:
        print(f"An error occurred during the API request: {e}")
        return None


####### TESTING ##################
# Example usage: gene found 
practice_ensembl = check_ensembl_location("BRCA1", False)
print(practice_ensembl)
# Example usage: gene not found
check_ensembl_location("InvalidGeneSymbol", False)


Gene 'BRCA1' found. HGNC ID: HGNC:1100, Name: BRCA1
{'hgnc_id': 'HGNC:1100', 'symbol': 'BRCA1', 'aliases': ['RNF53', 'BRCC1', 'PPP1R53', 'FANCS'], 'prev_symbols': [], 'entrez_id': '672', 'ensembl_id': 'ENSG00000012048', 'omim_ids': ['113705']}
Gene 'InvalidGeneSymbol' not found.


In [29]:
def ensembl_lookup_by_id(ensembl_id: str, url_hg19) -> dict | None:
    """Return Ensembl lookup record for an Ensembl stable ID (e.g., ENSG...)."""
    url = f"{ENSEMBL_REST_38}/lookup/id/{ensembl_id}"
    if url_hg19 == True: 
        url  = f"{ENSEMBL_REST_19}/lookup/id/{ensembl_id}"
    r = requests.get(url, headers=HEADERS, timeout=30)
    if r.status_code != 200:
        return None
    return r.json()

####### TESTING ##################
# Example usage: gene found 
practice_ensembl = ensembl_lookup_by_id("ENSG00000100342", True)
print(practice_ensembl)
# Example usage: gene not found
check_ensembl_location("InvalidGeneSymbol", False)

{'seq_region_name': '22', 'version': 16, 'assembly_name': 'GRCh37', 'biotype': 'protein_coding', 'species': 'homo_sapiens', 'id': 'ENSG00000100342', 'logic_name': 'ensembl_havana_gene_homo_sapiens_37', 'source': 'ensembl_havana', 'description': 'apolipoprotein L, 1 [Source:HGNC Symbol;Acc:618]', 'end': 36663576, 'db_type': 'core', 'canonical_transcript': 'ENST00000319136.4', 'strand': 1, 'display_name': 'APOL1', 'start': 36649056, 'object_type': 'Gene'}
Gene 'InvalidGeneSymbol' not found.


In [30]:
def check_many_hgnc_symbols(symbols):
    results = {}
    for s in symbols:
        info = check_hgnc_symbol(s)
        if info:
            results[s] = info
    return results


In [31]:
def check_many_ensembl_locations(symbols, url_19):
    results = {}
    for s in symbols:
        if url_19 == True: 
            info = ensembl_lookup_by_id(s, True)
        else: info = ensembl_lookup_by_id(s, False)
        if info:
            results[s] = info
    return results

#ensembl_locations_test = check_many_ensembl_locations("ENSG00000100342", True)
#print(ensembl_locations_test)

In [32]:
def ols_lookup_by_omim(omim_id):
    query = f"OMIM:{omim_id}"

    url = f"{OLS_BASE}/search"

    params = {
        "q": query,
        "rows": 10
    }

    r = requests.get(url, params=params)
    data = r.json()

    print(data)

    results = []

    for doc in data.get("response", {}).get("docs", []):
        results.append({
            "label": doc.get("label"),
            "ontology": doc.get("ontology_name"),
            "obo_id": doc.get("obo_id"),
            "iri": doc.get("iri")
        })

    return results

####### TESTING ##################
#disease_test = ols_lookup_by_omim(603743)
#print(disease_test)

In [33]:
def ols_lookup_multiple(symbols):
    results = {}
    for s in symbols:
        for omim_id in s:
            print(omim_id)
            info = ols_lookup_by_omim(omim_id)
            if info:
                results[omim_id] = info
    return results

####### TESTING ##################
#print(hgnc_df['omim_ids'])
#multiple_test = ols_lookup_multiple(hgnc_df['omim_ids'])

In [34]:
caps_only = re.findall(r'\b[A-Z]+\b', extracted_content)
print(caps_only)

['RESEARCH', 'V', 'M', 'B', 'W', 'C', 'C', 'M', 'J', 'L', 'C', 'M', 'R', 'N', 'NGS', 'ES', 'GS', 'EGBP', 'ES', 'GS', 'EGBP', 'ES', 'GS', 'ES', 'GS', 'RRAGD', 'ES', 'GS', 'EGBP', 'EGBP', 'NGS', 'RD', 'NGS', 'MGP', 'ES', 'GS', 'ES', 'EGBP', 'MGP', 'MN', 'USA', 'MN', 'USA', 'MN', 'USA', 'MN', 'USA', 'CA', 'USA', 'TX', 'USA', 'MN', 'ES', 'GS', 'MGP', 'MGP', 'EGBP', 'ES', 'MGP', 'ES', 'MGP', 'EGBP', 'GS', 'ES', 'ES', 'ES', 'ES', 'GS', 'EGBP', 'RD', 'MGP', 'EGBP', 'EGBP', 'ES', 'GS', 'ES', 'GS', 'EGBP', 'ES', 'GS', 'EGBP', 'CLIA', 'CAP', 'IRB', 'FASTQ', 'BAM', 'CRAM', 'VCF', 'SDMS', 'HIPAA', 'AI', 'VCF', 'BAM', 'HPO', 'ES', 'GS', 'EGBP', 'VUS', 'AR', 'IQR', 'FSGS', 'HGNC', 'G', 'G', 'ATT', 'RRAGD', 'HGNC', 'EGBP', 'HGNC', 'HGNC', 'MODY', 'CAKUT', 'MODY', 'CAKUT', 'HGNC', 'VUS', 'HPO', 'A', 'FSGS', 'RR', 'RR', 'RR', 'RR', 'RR', 'RR', 'FSGS', 'EGBP', 'ES', 'GS', 'NGS', 'ES', 'GS', 'VUS', 'LP', 'GUS', 'VUS', 'A', 'VUS', 'GUS', 'RRAGD', 'LP', 'G', 'A', 'VUS', 'HGNC', 'MIM', 'FSGS', 'A', 'FSGS', 

In [35]:
matches = re.findall(r'\b[A-Z][A-Z0-9]{2,}\b', extracted_content)
print(matches)

['RESEARCH', 'NGS', 'EGBP', 'EGBP', 'RRAGD', 'COL4A3', 'NPHS2', 'HNF1A', 'EGBP', 'EGBP', 'NGS', 'NGS', 'MGP', 'EGBP', 'MGP', 'USA', 'USA', 'USA', 'USA', 'USA', 'USA', 'MGP', 'MGP', 'EGBP', 'MGP', 'MGP', 'EGBP', 'EGBP', 'MGP', 'EGBP', 'EGBP', 'EGBP', 'EGBP', 'CLIA', 'CAP', 'IRB', 'FASTQ', 'BAM', 'CRAM', 'VCF', 'SDMS', 'HIPAA', 'VCF', 'BAM', 'HPO', 'EGBP', 'VUS', 'IQR', 'FSGS', 'APOL1', 'HGNC', 'ATT', 'RRAGD', 'HGNC', 'EGBP', 'COL4A3', 'HGNC', 'NPHS2', 'HGNC', 'MODY', 'CAKUT', 'MODY', 'CAKUT', 'HNF1A', 'HGNC', 'VUS', 'HPO', 'FSGS', 'FSGS', 'EGBP', 'NGS', 'VUS', 'GUS', 'VUS', 'COL4A3', 'VUS', 'NPHS2', 'GUS', 'RRAGD', 'HNF1A', 'VUS', 'COL4A3', 'HGNC', 'MIM', 'FSGS', 'FSGS', 'FSGS', 'EGBP', 'NPHS2', 'MIM', 'EGBP', 'NPHS2', 'HGNC', 'REVEL', 'EGBP', 'VUS', 'RRAGD', 'GTP', 'HGNC', 'REVEL', 'ECG', 'A1C', 'EGBP', 'HNF1A', 'EGBP', 'REVEL', 'MODY', 'NGS', 'VUS', 'MGP', 'APOL1', 'FSGS', 'APOL1', 'ESRD', 'EGPB', 'VUS', 'EGBP', 'COL4A3', 'HGNC', 'COL4A', 'COL4A3', 'NPHS2', 'HGNC', 'EGBP', 'NPHS2', 'H

In [36]:
filtered = [s for s in matches if likely_gene_symbol(s)]
print(filtered)

['RESEARCH', 'NGS', 'EGBP', 'EGBP', 'RRAGD', 'COL4A3', 'NPHS2', 'HNF1A', 'EGBP', 'EGBP', 'NGS', 'NGS', 'MGP', 'EGBP', 'MGP', 'USA', 'USA', 'USA', 'USA', 'USA', 'USA', 'MGP', 'MGP', 'EGBP', 'MGP', 'MGP', 'EGBP', 'EGBP', 'MGP', 'EGBP', 'EGBP', 'EGBP', 'EGBP', 'CLIA', 'CAP', 'IRB', 'FASTQ', 'BAM', 'CRAM', 'VCF', 'SDMS', 'HIPAA', 'VCF', 'BAM', 'HPO', 'EGBP', 'VUS', 'IQR', 'FSGS', 'APOL1', 'HGNC', 'ATT', 'RRAGD', 'HGNC', 'EGBP', 'COL4A3', 'HGNC', 'NPHS2', 'HGNC', 'MODY', 'CAKUT', 'MODY', 'CAKUT', 'HNF1A', 'HGNC', 'VUS', 'HPO', 'FSGS', 'FSGS', 'EGBP', 'NGS', 'VUS', 'GUS', 'VUS', 'COL4A3', 'VUS', 'NPHS2', 'GUS', 'RRAGD', 'HNF1A', 'VUS', 'COL4A3', 'HGNC', 'MIM', 'FSGS', 'FSGS', 'FSGS', 'EGBP', 'NPHS2', 'MIM', 'EGBP', 'NPHS2', 'HGNC', 'REVEL', 'EGBP', 'VUS', 'RRAGD', 'GTP', 'HGNC', 'REVEL', 'ECG', 'A1C', 'EGBP', 'HNF1A', 'EGBP', 'REVEL', 'MODY', 'NGS', 'VUS', 'MGP', 'APOL1', 'FSGS', 'APOL1', 'ESRD', 'EGPB', 'VUS', 'EGBP', 'COL4A3', 'HGNC', 'COL4A', 'COL4A3', 'NPHS2', 'HGNC', 'EGBP', 'NPHS2', 'H

In [37]:
STOP = {"DNA","RNA","ATP","GTP","FASTQ","VCF","NGS","HGNC","HPO","HIPAA","CLIA","IRB","USA","RESEARCH","VUS"}

candidates = [s for s in filtered
              if re.fullmatch(r"[A-Z][A-Z0-9]{2,11}", str(s).strip())
              and str(s).strip() not in STOP]


In [38]:
candidates = sorted(set(candidates))
print(candidates)

['A1C', 'A1ED8', 'ACMG', 'APOL1', 'ATT', 'B7987', 'BAM', 'C4D2C', 'CAKUT', 'CAL', 'CAP', 'CIM', 'COL4A', 'COL4A3', 'COL4A4', 'CRAM', 'CRR', 'E3570', 'ECG', 'EGBP', 'EGPB', 'EMHF', 'EMM', 'ESRD', 'FE1AB', 'FEE99', 'FFBE7', 'FSGS', 'GUS', 'HERC2', 'HNF1A', 'IQR', 'JASN', 'JFM', 'JKJ', 'MGP', 'MHT', 'MIM', 'MJV', 'MODY', 'NPHS2', 'R229Q', 'REVEL', 'RRAGD', 'S1098', 'SDMS']


In [39]:
found_genes = check_many_hgnc_symbols(candidates)
#found_genes = check_hgnc_symbol(candidates)

print("here are the found genes") 
print(found_genes)

Gene 'A1C' not found.
Gene 'A1ED8' not found.
Gene 'ACMG' found. HGNC ID: HGNC:116, Name: ACMG
Gene 'APOL1' found. HGNC ID: HGNC:618, Name: APOL1
Gene 'ATT' not found.
Gene 'B7987' not found.
Gene 'BAM' not found.
Gene 'C4D2C' not found.
Gene 'CAKUT' not found.
Gene 'CAL' not found.
Gene 'CAP' not found.
Gene 'CIM' not found.
Gene 'COL4A' not found.
Gene 'COL4A3' found. HGNC ID: HGNC:2204, Name: COL4A3
Gene 'COL4A4' found. HGNC ID: HGNC:2206, Name: COL4A4
Gene 'CRAM' not found.
Gene 'CRR' not found.
Gene 'E3570' not found.
Gene 'ECG' not found.
Gene 'EGBP' not found.
Gene 'EGPB' not found.
Gene 'EMHF' not found.
Gene 'EMM' not found.
Gene 'ESRD' not found.
Gene 'FE1AB' not found.
Gene 'FEE99' not found.
Gene 'FFBE7' not found.
Gene 'FSGS' not found.
Gene 'GUS' not found.
Gene 'HERC2' found. HGNC ID: HGNC:4868, Name: HERC2
Gene 'HNF1A' found. HGNC ID: HGNC:11621, Name: HNF1A
Gene 'IQR' not found.
Gene 'JASN' not found.
Gene 'JFM' not found.
Gene 'JKJ' not found.
Gene 'MGP' found. HGNC I

In [40]:
for gene_symbol, gene_data in found_genes.items():
    print(gene_symbol, gene_data['hgnc_id'], gene_data['aliases'],gene_data['prev_symbols'], gene_data['ensembl_id'], gene_data['omim_ids'])

ACMG HGNC:116 [] [] None []
APOL1 HGNC:618 [] ['APOL'] ENSG00000100342 ['603743']
COL4A3 HGNC:2204 [] [] ENSG00000169031 ['120070']
COL4A4 HGNC:2206 ['CA44'] [] ENSG00000081052 ['120131']
HERC2 HGNC:4868 ['jdf2', 'p528', 'D15F37S1'] [] ENSG00000128731 ['605837']
HNF1A HGNC:11621 ['HNF1', 'LFB1', 'HNF1α'] ['MODY3', 'TCF1'] ENSG00000135100 ['142410']
MGP HGNC:7060 [] [] ENSG00000111341 ['154870']
NPHS2 HGNC:13394 ['SRN1', 'PDCN'] [] ENSG00000116218 ['604766']
RRAGD HGNC:19903 ['DKFZP761H171', 'bA11D8.2.1'] [] ENSG00000025039 ['608268']


In [41]:
hgnc_rows = []
for gene, data in found_genes.items():
    hgnc_rows.append({
        "symbol": gene,
        **data
    })
#print(rows)

hgnc_df = pd.DataFrame(hgnc_rows)

hgnc_df_filtered = hgnc_df[['symbol', 'hgnc_id', 'aliases','omim_ids', 'ensembl_id']].copy()

hgnc_df_filtered['omim_ids'] = hgnc_df_filtered['omim_ids'].str[0]

#print(hgnc_df)

print(hgnc_df_filtered )

   symbol     hgnc_id                     aliases omim_ids       ensembl_id
0    ACMG    HGNC:116                          []      NaN             None
1   APOL1    HGNC:618                          []   603743  ENSG00000100342
2  COL4A3   HGNC:2204                          []   120070  ENSG00000169031
3  COL4A4   HGNC:2206                      [CA44]   120131  ENSG00000081052
4   HERC2   HGNC:4868      [jdf2, p528, D15F37S1]   605837  ENSG00000128731
5   HNF1A  HGNC:11621         [HNF1, LFB1, HNF1α]   142410  ENSG00000135100
6     MGP   HGNC:7060                          []   154870  ENSG00000111341
7   NPHS2  HGNC:13394                [SRN1, PDCN]   604766  ENSG00000116218
8   RRAGD  HGNC:19903  [DKFZP761H171, bA11D8.2.1]   608268  ENSG00000025039


In [42]:
ensembl_genes_38 = check_many_ensembl_locations(hgnc_df['ensembl_id'], False)


ensembl_genes_19 = check_many_ensembl_locations(hgnc_df['ensembl_id'], True)

#print(ensembl_genes_38)

#print(ensembl_genes_19)

In [43]:
ensembl_rows_38 = []
for gene, data in ensembl_genes_38.items():
    ensembl_rows_38.append({
        "symbol": gene,
        **data
    })
#print(rows)

ensembl_38_df = pd.DataFrame(ensembl_rows_38)

ensembl_38_df_filtered = ensembl_38_df[['symbol', 'start', 'end', 'seq_region_name']].rename(columns = {'symbol': 'ensembl_id',
                                                                                                     'start': 'hg38_start',
                                                                                                     'end': 'hg38_end', 
                                                                                                     'seq_region_name': 'chrom_number'})

print(ensembl_38_df_filtered)

        ensembl_id  hg38_start   hg38_end chrom_number
0  ENSG00000100342    36253071   36267530           22
1  ENSG00000169031   227164624  227314792            2
2  ENSG00000081052   227002714  227164453            2
3  ENSG00000128731    28111040   28322179           15
4  ENSG00000135100   120978543  121002512           12
5  ENSG00000111341    14880864   14887639           12
6  ENSG00000116218   179550494  179575952            1
7  ENSG00000025039    89364616   89412735            6


In [44]:
ensembl_rows_19 = []
for gene, data in ensembl_genes_19.items():
    ensembl_rows_19.append({
        "symbol": gene,
        **data
    })
#print(rows)

ensembl_19_df = pd.DataFrame(ensembl_rows_19)

ensembl_19_df_filtered = ensembl_38_df[['symbol', 'start', 'end', 'seq_region_name']].rename(columns = {'symbol': 'ensembl_id',
                                                                                                     'start': 'hg19_start',
                                                                                                     'end': 'hg19_end', 
                                                                                                     'seq_region_name': 'chrom_number'})

print(ensembl_19_df_filtered)

        ensembl_id  hg19_start   hg19_end chrom_number
0  ENSG00000100342    36253071   36267530           22
1  ENSG00000169031   227164624  227314792            2
2  ENSG00000081052   227002714  227164453            2
3  ENSG00000128731    28111040   28322179           15
4  ENSG00000135100   120978543  121002512           12
5  ENSG00000111341    14880864   14887639           12
6  ENSG00000116218   179550494  179575952            1
7  ENSG00000025039    89364616   89412735            6


In [45]:
found_disease = ols_lookup_multiple(hgnc_df['omim_ids'])

#print(found_disease)

603743
{'response': {'docs': [{'iri': 'http://www.orpha.net/ORDO/Orphanet_240672', 'ontology_name': 'ordo', 'ontology_prefix': 'ORDO', 'short_form': 'ORDO_240672', 'description': [], 'label': 'apolipoprotein L1', 'obo_id': 'ORDO:240672', 'type': 'class'}, {'iri': 'http://purl.obolibrary.org/obo/DOID_384', 'ontology_name': 'doid', 'ontology_prefix': 'DOID', 'short_form': 'DOID_384', 'description': ['OMIM mapping confirmed by DO. [LS]. OMIM mapping confirmed by DO. [SN].'], 'label': 'Wolff-Parkinson-White syndrome', 'obo_id': 'DOID:384', 'type': 'class', 'exact_synonyms': ['Anomalous A-V excitation', 'Wolff-Parkinson-White pattern', 'anomalous atrioventricular excitation']}, {'iri': 'http://purl.obolibrary.org/obo/MONDO_0009421', 'ontology_name': 'mondo', 'related_synonyms': ['hypogonadism and testicular atrophy'], 'ontology_prefix': 'MONDO', 'short_form': 'MONDO_0009421', 'description': ['Editor note: check OMIM'], 'label': 'hypogonadism, male', 'obo_id': 'MONDO:0009421', 'type': 'class

In [46]:
disease_row = []

for gene, disease_list in found_disease.items():
    for disease in disease_list:   # <-- loop through list
        disease_row.append({
            "omim_ids": gene,
            **disease
        })

disease_df = pd.DataFrame(disease_row)

disease_df_filtered = disease_df[['omim_ids', 'label']].rename(columns={'label': 'disease'})

print(disease_df_filtered)

   omim_ids                                            disease
0    603743                                  apolipoprotein L1
1    603743                     Wolff-Parkinson-White syndrome
2    603743                                 hypogonadism, male
3    603743                                     gyrate atrophy
4    603743       growth hormone secreting pituitary adenoma 1
..      ...                                                ...
75   608268       growth hormone secreting pituitary adenoma 1
76   608268  basal ganglia calcification, idiopathic, child...
77   608268  amyotrophic lateral sclerosis with polyglucosa...
78   608268          pregnancy loss, recurrent, susceptibility
79   608268  Autosomal recessive non-syndromic intellectual...

[80 rows x 2 columns]


In [48]:
baylor_df = pd.merge(hgnc_df_filtered, ensembl_19_df_filtered, on="ensembl_id")
baylor_df = pd.merge(baylor_df, ensembl_38_df_filtered, on="ensembl_id")
baylor_df = pd.merge(baylor_df, disease_df_filtered, on="omim_ids")
baylor_df_final = baylor_df[['symbol', 'hgnc_id', 'aliases', 'hg38_start', 'hg38_end',
                             'hg19_start', 'hg19_end', 'chrom_number_x','disease']]
print(baylor_df_final)

   symbol     hgnc_id                     aliases  hg38_start  hg38_end  \
0   APOL1    HGNC:618                          []    36253071  36267530   
1   APOL1    HGNC:618                          []    36253071  36267530   
2   APOL1    HGNC:618                          []    36253071  36267530   
3   APOL1    HGNC:618                          []    36253071  36267530   
4   APOL1    HGNC:618                          []    36253071  36267530   
..    ...         ...                         ...         ...       ...   
75  RRAGD  HGNC:19903  [DKFZP761H171, bA11D8.2.1]    89364616  89412735   
76  RRAGD  HGNC:19903  [DKFZP761H171, bA11D8.2.1]    89364616  89412735   
77  RRAGD  HGNC:19903  [DKFZP761H171, bA11D8.2.1]    89364616  89412735   
78  RRAGD  HGNC:19903  [DKFZP761H171, bA11D8.2.1]    89364616  89412735   
79  RRAGD  HGNC:19903  [DKFZP761H171, bA11D8.2.1]    89364616  89412735   

    hg19_start  hg19_end chrom_number_x  \
0     36253071  36267530             22   
1     3625307

In [49]:
baylor_df_final.to_csv("baylor_dataframe.csv", index=False)

In [ ]:
# Create the database and tables by running create_mydb script

%run create_mydb.ipynb